In [2]:
import torch
# Load the dataset back with tensors intact
PROF_prefinal_dataset = torch.load(
    r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\prof\PROF_prefinal_dataset.pt"
)


In [3]:
import torch
# Load the dataset back with tensors intact
PHD_prefinal_dataset = torch.load(
    r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\phd\PHD_prefinal_dataset.pt"
)


In [6]:
PHD_prefinal_dataset[0]

{'scholar_id': 150,
 'cat_ids': tensor([117,   8,   7,   1,   4]),
 'cat_emb': tensor([-0.0536, -0.1200, -0.0054,  0.0758, -0.0147,  0.0565, -0.0610, -0.0036,
         -0.0930,  0.0555, -0.0396,  0.0412, -0.0317, -0.0834,  0.0974, -0.1482,
         -0.0185,  0.0415, -0.1815, -0.1261,  0.0252, -0.0825, -0.0374,  0.0390,
          0.0032, -0.0658,  0.0674,  0.0053, -0.0444,  0.0385, -0.0905, -0.0797,
          0.0449,  0.1074,  0.0917, -0.0215,  0.0486,  0.0277,  0.0064,  0.0375,
         -0.0416,  0.0087, -0.0865,  0.0190, -0.1329, -0.0959,  0.0765,  0.0646,
          0.0882,  0.0114,  0.1502, -0.0247, -0.0753, -0.0546, -0.0567, -0.0513,
          0.0158, -0.0493,  0.1088, -0.0347,  0.0529,  0.0226,  0.0123,  0.0998,
          0.1051,  0.1086,  0.0592,  0.0393, -0.0944,  0.0304, -0.0555,  0.1885,
         -0.0316,  0.0072,  0.0061, -0.0697,  0.0126, -0.0754, -0.0975,  0.0199,
         -0.0057, -0.0091, -0.0711, -0.0760, -0.0594, -0.0393,  0.1275, -0.0041,
          0.0191, -0.0218,  0.0

In [4]:
# Stack text embeddings for scholars: Shape (num_scholars, 384)
q_matrix = torch.stack([item["text_emb"] for item in PHD_prefinal_dataset])

# Stack text embeddings for professors: Shape (num_profs, 384)
cand_matrix = torch.stack([item["text_emb"] for item in PROF_prefinal_dataset])


In [5]:
from dataclasses import dataclass
import torch

@dataclass
class LabeledPair:
    scholar_id: int
    prof_id: int
    label: float
    similarity_score: float
    phd_text_emb: torch.Tensor
    prof_text_emb: torch.Tensor
    phd_cat_emb: torch.Tensor
    prof_cat_emb: torch.Tensor
    phd_num_emb: torch.Tensor

    def __repr__(self):
        return (
            f"LabeledPair(scholar_id={self.scholar_id}, prof_id={self.prof_id}, "
            f"label={self.label}, sim_score={self.similarity_score:.4f})"
        )


In [6]:
import torch

def exact_knn_search(query_vectors: torch.Tensor, index_vectors: torch.Tensor, k: int = 100):
    """Performs exact KNN search using matrix multiplication.
    Returns:
        sim_matrix: Full similarity scores matrix
        top_k_scores: Top-k scores per query
        top_k_indices: Top-k candidate indices per query
    """
    sim_matrix = torch.matmul(query_vectors, index_vectors.T)
    top_k_scores, top_k_indices = torch.topk(sim_matrix, k=k, dim=1, largest=True, sorted=True)
    return sim_matrix, top_k_scores, top_k_indices


In [7]:
import random

def generate_training_pairs(
    phd_dataset, prof_dataset, top_k_indices, sim_matrix, num_hard=3, num_random=5
):
    """Generates 2 positives, 3 hard negatives, and 5 random negatives per scholar

    along with their similarity scores and text/categorical/numerical embeddings.
    """
    labeled_dataset = []
    num_professors = len(prof_dataset)
    all_prof_indices = list(range(num_professors))

    # Convert indices to CPU list for easy iteration
    top_k_list = top_k_indices.cpu().tolist()

    for i, scholar_item in enumerate(phd_dataset):
        scholar_id = scholar_item["scholar_id"]
        scholar_top_k = top_k_list[i]

        # Extract student embeddings (reused across all 10 pairs for this student)
        phd_text_emb = scholar_item.get("text_emb")
        phd_cat_emb = scholar_item.get("cat_emb")
        phd_num_emb = scholar_item.get("num_emb")

        # Inner helper to build a LabeledPair cleanly
        def create_pair(prof_idx, label):
            prof_item = prof_dataset[prof_idx]
            prof_id = prof_item["prof_id"]

            # Index the similarity score from our similarity matrix
            similarity_score = sim_matrix[i, prof_idx].item()

            # Extract professor embeddings (candidate side uses 'cate_emb' for categorical)
            prof_text_emb = prof_item.get("text_emb")
            prof_cat_emb = prof_item.get("cate_emb")

            return LabeledPair(
                scholar_id=scholar_id,
                prof_id=prof_id,
                label=label,
                similarity_score=similarity_score,
                phd_text_emb=phd_text_emb,
                prof_text_emb=prof_text_emb,
                phd_cat_emb=phd_cat_emb,
                prof_cat_emb=prof_cat_emb,
                phd_num_emb=phd_num_emb,
            )

        # 1. Positives (Top 2 similarity ranks)
        pos_indices = scholar_top_k[0:2]
        for idx in pos_indices:
            labeled_dataset.append(create_pair(idx, 1.0))

        # 2. Hard Negatives (3 random samples from ranks 20 to 100)
        hard_neg_candidate_indices = scholar_top_k[19:100]
        hard_neg_indices = random.sample(hard_neg_candidate_indices, num_hard)
        for idx in hard_neg_indices:
            labeled_dataset.append(create_pair(idx, 0.0))

        # 3. Random Negatives (5 random samples from all profs, excluding the top 100)
        exclude_set = set(scholar_top_k)
        allowed_random_indices = [
            idx for idx in all_prof_indices if idx not in exclude_set
        ]

        random_neg_indices = random.sample(allowed_random_indices, num_random)
        for idx in random_neg_indices:
            labeled_dataset.append(create_pair(idx, 0.0))

    return labeled_dataset


In [8]:
# 1. Run exact KNN search (returns sim_matrix, scores, and indices)
sim_matrix, scores, indices = exact_knn_search(q_matrix, cand_matrix, k=100)

# 2. Generate the labeled training pairs with all embeddings and similarity scores
labeled_pairs = generate_training_pairs(
    phd_dataset=PHD_prefinal_dataset,
    prof_dataset=PROF_prefinal_dataset,
    top_k_indices=indices,
    sim_matrix=sim_matrix
)

# 3. Verify the result
print(f"Total labeled pairs: {len(labeled_pairs)}")
first_pair = labeled_pairs[0]
print("\nFirst pair details:")
print(first_pair)
print("phd_text_emb shape:", first_pair.phd_text_emb.shape)
print("prof_text_emb shape:", first_pair.prof_text_emb.shape)
print("phd_cat_emb shape:", first_pair.phd_cat_emb.shape)
print("prof_cat_emb shape:", first_pair.prof_cat_emb.shape)
print("phd_num_emb shape:", first_pair.phd_num_emb.shape)


Total labeled pairs: 8520

First pair details:
LabeledPair(scholar_id=150, prof_id=4304, label=1.0, sim_score=0.5398)
phd_text_emb shape: torch.Size([384])
prof_text_emb shape: torch.Size([384])
phd_cat_emb shape: torch.Size([160])
prof_cat_emb shape: torch.Size([160])
phd_num_emb shape: torch.Size([32])


In [8]:
print(labeled_pairs[0])
# Output: LabeledPair(scholar_id=150, prof_id=102, label=1.0, sim_score=0.8121)


LabeledPair(scholar_id=150, prof_id=4304, label=1.0, sim_score=0.5398)


In [9]:
len(labeled_pairs)


8520

In [10]:
# Access basic fields
print("Scholar ID:", labeled_pairs[0].scholar_id)
print("Professor ID:", labeled_pairs[0].prof_id)
print("Label (Positive/Negative):", labeled_pairs[0].label)
print("Similarity Score:", labeled_pairs[0].similarity_score)

# Access embedding tensors
print("Student Text Embedding shape:", labeled_pairs[0].phd_text_emb.shape)


Scholar ID: 150
Professor ID: 4304
Label (Positive/Negative): 1.0
Similarity Score: 0.53983473777771
Student Text Embedding shape: torch.Size([384])


In [11]:
# Select the first pair
first_pair = labeled_pairs[0]

# 1. Print the shape of the embeddings to verify their existence
print("Student Text Embedding shape:", first_pair.phd_text_emb.shape)
print("Student Categorical Embedding shape:", first_pair.phd_cat_emb.shape)
print("Professor Text Embedding shape:", first_pair.prof_text_emb.shape)

# 2. View the actual raw tensor values of the first 5 dimensions of the student text embedding
print("\nFirst 5 values of student text embedding:")
print(first_pair.phd_text_emb[:5])


Student Text Embedding shape: torch.Size([384])
Student Categorical Embedding shape: torch.Size([160])
Professor Text Embedding shape: torch.Size([384])

First 5 values of student text embedding:
tensor([-0.0208, -0.0029, -0.0522,  0.0105,  0.0925])


In [9]:
import os
import torch

target_dir = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\final_dataset"
os.makedirs(target_dir, exist_ok=True)
pt_path = os.path.join(target_dir, "labeled_training_dataset.pt")

print("Stacking tensors for fast serialization...")

# Stack all features into single large tensors (done instantly in C++)
fast_dataset = {
    "scholar_id": torch.tensor([p.scholar_id for p in labeled_pairs], dtype=torch.long),
    "prof_id": torch.tensor([p.prof_id for p in labeled_pairs], dtype=torch.long),
    "label": torch.tensor([p.label for p in labeled_pairs], dtype=torch.float32),
    "similarity_score": torch.tensor([p.similarity_score for p in labeled_pairs], dtype=torch.float32),
    
    # 2D Tensors: Shape (num_pairs, embedding_dim)
    "phd_text_emb": torch.stack([p.phd_text_emb for p in labeled_pairs]),
    "prof_text_emb": torch.stack([p.prof_text_emb for p in labeled_pairs]),
    "phd_cat_emb": torch.stack([p.phd_cat_emb for p in labeled_pairs]),
    "prof_cat_emb": torch.stack([p.prof_cat_emb for p in labeled_pairs]),
    "phd_num_emb": torch.stack([p.phd_num_emb for p in labeled_pairs])
}

print(f"Saving to {pt_path}...")
torch.save(fast_dataset, pt_path)
print("Done saving!")


Stacking tensors for fast serialization...
Saving to C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\final_dataset\labeled_training_dataset.pt...
Done saving!


In [10]:
import torch

pt_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\final_dataset\labeled_training_dataset.pt"

print("Loading dataset...")
loaded_dataset = torch.load(pt_path)

# Verify the shapes
print("Loaded successfully!")
print("Number of training pairs:", len(loaded_dataset["label"]))
print("PhD Text Embeddings Matrix Shape:", loaded_dataset["phd_text_emb"].shape) 
# Example output shape: (num_pairs, 384)


Loading dataset...
Loaded successfully!
Number of training pairs: 8520
PhD Text Embeddings Matrix Shape: torch.Size([8520, 384])


In [ ]:
# How to use this for PyTorch Training (Optional but helpful)
# This stacked tensor format makes it extremely easy and fast to build a standard PyTorch Dataset for training your model:

from torch.utils.data import Dataset, DataLoader

class MatchingDataset(Dataset):
    def __init__(self, data_dict):
        self.data = data_dict

    def __len__(self):
        return len(self.data["label"])

    def __getitem__(self, idx):
        return {
            "scholar_id": self.data["scholar_id"][idx],
            "prof_id": self.data["prof_id"][idx],
            "label": self.data["label"][idx],
            "similarity_score": self.data["similarity_score"][idx],
            "phd_text_emb": self.data["phd_text_emb"][idx],
            "prof_text_emb": self.data["prof_text_emb"][idx],
            "phd_cat_emb": self.data["phd_cat_emb"][idx],
            "prof_cat_emb": self.data["prof_cat_emb"][idx],
            "phd_num_emb": self.data["phd_num_emb"][idx]
        }

# Create Dataset and DataLoader
dataset = MatchingDataset(loaded_dataset)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True)


In [ ]:
# loaded_dataset


{'scholar_id': tensor([ 150,  150,  150,  ..., 1001, 1001, 1001]),
 'prof_id': tensor([4304, 3970, 4255,  ..., 4614, 4419, 4335]),
 'label': tensor([1., 1., 0.,  ..., 0., 0., 0.]),
 'similarity_score': tensor([ 0.5398,  0.5374,  0.4072,  ..., -0.1018,  0.2661,  0.0353]),
 'phd_text_emb': tensor([[-0.0208, -0.0029, -0.0522,  ...,  0.0218, -0.0574, -0.0424],
         [-0.0208, -0.0029, -0.0522,  ...,  0.0218, -0.0574, -0.0424],
         [-0.0208, -0.0029, -0.0522,  ...,  0.0218, -0.0574, -0.0424],
         ...,
         [-0.0544, -0.1067, -0.0244,  ..., -0.0772, -0.0343, -0.0965],
         [-0.0544, -0.1067, -0.0244,  ..., -0.0772, -0.0343, -0.0965],
         [-0.0544, -0.1067, -0.0244,  ..., -0.0772, -0.0343, -0.0965]]),
 'prof_text_emb': tensor([[-4.2198e-02,  1.2413e-02, -7.9653e-03,  ..., -7.0010e-04,
           5.9135e-05,  6.1200e-02],
         [ 3.0454e-02, -3.9997e-02, -3.0613e-03,  ...,  5.4363e-03,
          -1.0399e-01, -2.9855e-02],
         [-1.2941e-02, -6.1422e-03,  1.8997